In [33]:
import numpy as np
import pandas as pd
from finance_byu.summarize import summary
from scipy import stats



In [41]:
def load_and_process(filepath, n_industries):
    # load
    df = pd.read_csv(filepath, na_values=[-99.99])
    
    # clean dates and column names
    df.columns = df.columns.str.lower()
    df['date'] = pd.to_datetime(df['date'], format='%Y%m').dt.to_period('M')

    # filter to paper's sample period
    df = df[(df['date'] >= '1963-07') & (df['date'] <= '1995-07')]
    
    # unpivot
    df = df.melt(id_vars='date', var_name='industry', value_name='industry_return')
    
    # calculate momentum
    # use 6 month rolling window and simple returns as used in paper
    df['mom'] = (df.groupby('industry')['industry_return']
                   .transform(lambda x: (1 + x/100).shift(1)
                   .rolling(6).apply(lambda r: r.prod() - 1)))
    
    # rank industries each month
    df['rank'] = df.groupby('date')['mom'].rank(ascending=False)
    
    # label winners and losers
    df['portfolio'] = 'middle'
    df.loc[df['rank'] <= 3, 'portfolio'] = 'winner'
    df.loc[df['rank'] > n_industries - 3, 'portfolio'] = 'loser'
    
    return df

# run for all three
df17 = load_and_process('17_Industry_Portfolios.csv', 17)
df30 = load_and_process('30_Industry_Portfolios.csv', 30)
df48 = load_and_process('48_Industry_Portfolios.csv', 48)

In [35]:
df17

,date,industry,industry_return,mom,rank,portfolio
0,1963-07,food,-0.15,NaN,NaN,middle
1,1963-08,food,4.60,NaN,NaN,middle
2,1963-09,food,-1.26,NaN,NaN,middle
3,1963-10,food,2.09,NaN,NaN,middle
4,1963-11,food,-0.47,NaN,NaN,middle
...,...,...,...,...,...,...
6540,1995-03,other,2.31,0.041579,5.0,middle
6541,1995-04,other,1.86,0.071425,9.0,middle
6542,1995-05,other,1.06,0.079372,10.0,middle
6543,1995-06,other,5.28,0.136620,11.0,middle


In [42]:
df30

,date,industry,industry_return,mom,rank,portfolio
0,1963-07,food,0.04,NaN,NaN,middle
1,1963-08,food,4.82,NaN,NaN,middle
2,1963-09,food,-1.43,NaN,NaN,middle
3,1963-10,food,2.41,NaN,NaN,middle
4,1963-11,food,-0.46,NaN,NaN,middle
...,...,...,...,...,...,...
11545,1995-03,other,3.39,-0.029986,22.0,middle
11546,1995-04,other,1.34,0.039057,20.0,middle
11547,1995-05,other,4.65,0.041420,21.0,middle
11548,1995-06,other,2.53,0.161140,17.0,middle


In [43]:
df48

,date,industry,industry_return,mom,rank,portfolio
0,1963-07,agric,3.04,NaN,NaN,middle
1,1963-08,agric,-0.32,NaN,NaN,middle
2,1963-09,agric,-1.94,NaN,NaN,middle
3,1963-10,agric,-0.13,NaN,NaN,middle
4,1963-11,agric,-3.06,NaN,NaN,middle
...,...,...,...,...,...,...
18475,1995-03,other,3.21,-0.093782,45.0,middle
18476,1995-04,other,0.15,-0.042478,44.0,middle
18477,1995-05,other,5.08,-0.038735,43.0,middle
18478,1995-06,other,4.09,0.105502,36.0,middle


In [44]:
# compute winners - losers as done with the paper
def compute_wml(df):
    monthly = df.groupby(['date', 'portfolio'])['industry_return'].mean().unstack()
    monthly['wml'] = monthly['winner'] - monthly['loser']
    return monthly

wml17 = compute_wml(df17)
wml30 = compute_wml(df30)
wml48 = compute_wml(df48)

In [39]:
wml17

portfolio,loser,middle,winner,wml
date,,,,
1963-07,NaN,-0.587059,NaN,NaN
1963-08,NaN,5.462353,NaN,NaN
1963-09,NaN,-1.654706,NaN,NaN
1963-10,NaN,2.735882,NaN,NaN
1963-11,NaN,-0.625294,NaN,NaN
...,...,...,...,...
1995-03,4.720000,2.751818,3.126667,-1.593333
1995-04,0.116667,1.385455,4.946667,4.830000
1995-05,2.260000,3.509091,2.926667,0.666667


In [45]:
# compute mean monthly WML returns and t-statistics for each industry classification
# paper target: mean = 0.43% per month, t-stat = 4.24 (using 20 industries, 1963-1995)

for name, wml in [('17', wml17), ('30', wml30), ('48', wml48)]:
    mean = wml['wml'].mean()
    tstat = stats.ttest_1samp(wml['wml'].dropna(), 0).statistic
    print(f'{name} industries: mean = {mean:.4f}, t-stat = {tstat:.2f}')

17 industries: mean = 0.3911, t-stat = 1.95
30 industries: mean = 0.7466, t-stat = 3.09
48 industries: mean = 0.8781, t-stat = 2.91
